In [1]:
import pandas as pd

network_df = pd.read_csv("network_measurements.csv")
areas_df = pd.read_csv("kithul_areas.csv")

print("Network dataset:")
display(network_df)

print("Kithul areas:")
display(areas_df)

Network dataset:


,area_name,district,lat,lon,avg_d_kbps,avg_u_kbps,avg_lat_ms,tower_count,nearest_tower_km
0,Neluwa,Galle,6.367,80.367,4200,900,140,4,2.1
1,Warukandeniya,Galle,6.350,80.390,1800,500,230,2,4.7
2,Tinniyawala,Kalutara,6.450,80.170,6500,1200,80,5,1.5


Kithul areas:


,area_name,district,lat,lon
0,Neluwa,Galle,6.367,80.367
1,Warukandeniya,Galle,6.350,80.390
2,Tinniyawala,Kalutara,6.450,80.170
3,Kalawana,Ratnapura,6.533,80.400
4,Yatiyanthota,Kegalle,7.020,80.300


In [2]:
def classify_network(row):
    download = row["avg_d_kbps"]
    upload = row["avg_u_kbps"]
    latency = row["avg_lat_ms"]

    if download >= 5000 and upload >= 1000 and latency <= 100:
        return "Good"
    elif download >= 1000 and latency <= 250:
        return "Moderate"
    else:
        return "Poor"

network_df["network_suitability"] = network_df.apply(classify_network, axis=1)

network_df["sync_strategy"] = network_df["network_suitability"].map({
    "Good": "Full Sync",
    "Moderate": "Priority Sync Only",
    "Poor": "Offline Retry"
})

display(network_df)

,area_name,district,lat,lon,avg_d_kbps,avg_u_kbps,avg_lat_ms,tower_count,nearest_tower_km,network_suitability,sync_strategy
0,Neluwa,Galle,6.367,80.367,4200,900,140,4,2.1,Moderate,Priority Sync Only
1,Warukandeniya,Galle,6.350,80.390,1800,500,230,2,4.7,Moderate,Priority Sync Only
2,Tinniyawala,Kalutara,6.450,80.170,6500,1200,80,5,1.5,Good,Full Sync


In [18]:
from sklearn.ensemble import RandomForestClassifier
import joblib

features = [
    "lat",
    "lon",
    "avg_d_kbps",
    "avg_u_kbps",
    "avg_lat_ms",
    "tower_count",
    "nearest_tower_km"
]

X = network_df[features]
y = network_df["network_suitability"]

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X, y)

print("Model trained successfully")

Model trained successfully


In [5]:
def recommend_sync_strategy(network_class):
    if network_class == "Good":
        return "Full Sync - sync all pending records"
    elif network_class == "Moderate":
        return "Priority Sync Only - sync urgent collection records first"
    else:
        return "Offline Retry - save locally and retry later"


def predict_for_area(area_name):
    area_rows = network_df[
        network_df["area_name"].str.lower() == area_name.lower()
    ]

    if area_rows.empty:
        return f"No network measurement data found for {area_name}. Add this area to network_measurements.csv first."

    sample = area_rows.tail(1)

    predicted_class = model.predict(sample[features])[0]
    confidence = model.predict_proba(sample[features]).max() * 100
    strategy = recommend_sync_strategy(predicted_class)

    result = {
        "Area": area_name,
        "Predicted Network Suitability": predicted_class,
        "Confidence": f"{confidence:.2f}%",
        "Recommended Sync Strategy": strategy,
        "Download Speed kbps": int(sample["avg_d_kbps"].iloc[0]),
        "Upload Speed kbps": int(sample["avg_u_kbps"].iloc[0]),
        "Latency ms": int(sample["avg_lat_ms"].iloc[0]),
        "Tower Count": int(sample["tower_count"].iloc[0]),
        "Nearest Tower km": float(sample["nearest_tower_km"].iloc[0])
    }

    return pd.DataFrame([result])

In [6]:
new_rows = pd.DataFrame([
    {
        "area_name": "Kalawana",
        "district": "Ratnapura",
        "lat": 6.533,
        "lon": 80.400,
        "avg_d_kbps": 3500,
        "avg_u_kbps": 750,
        "avg_lat_ms": 170,
        "tower_count": 3,
        "nearest_tower_km": 3.2
    },
    {
        "area_name": "Yatiyanthota",
        "district": "Kegalle",
        "lat": 7.020,
        "lon": 80.300,
        "avg_d_kbps": 5200,
        "avg_u_kbps": 1000,
        "avg_lat_ms": 100,
        "tower_count": 4,
        "nearest_tower_km": 2.0
    }
])

network_df = pd.concat([network_df, new_rows], ignore_index=True)

network_df["network_suitability"] = network_df.apply(classify_network, axis=1)

network_df["sync_strategy"] = network_df["network_suitability"].map({
    "Good": "Full Sync",
    "Moderate": "Priority Sync Only",
    "Poor": "Offline Retry"
})

display(network_df)

,area_name,district,lat,lon,avg_d_kbps,avg_u_kbps,avg_lat_ms,tower_count,nearest_tower_km,network_suitability,sync_strategy
0,Neluwa,Galle,6.367,80.367,4200,900,140,4,2.1,Moderate,Priority Sync Only
1,Warukandeniya,Galle,6.350,80.390,1800,500,230,2,4.7,Moderate,Priority Sync Only
2,Tinniyawala,Kalutara,6.450,80.170,6500,1200,80,5,1.5,Good,Full Sync
3,Kalawana,Ratnapura,6.533,80.400,3500,750,170,3,3.2,Moderate,Priority Sync Only
4,Yatiyanthota,Kegalle,7.020,80.300,5200,1000,100,4,2.0,Good,Full Sync


In [7]:
X = network_df[features]
y = network_df["network_suitability"]

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X, y)

print("Model retrained successfully with updated data")

Model retrained successfully with updated data


In [8]:
new_rows = pd.DataFrame([
    {
        "area_name": "Kalawana",
        "district": "Ratnapura",
        "lat": 6.533,
        "lon": 80.400,
        "avg_d_kbps": 3500,
        "avg_u_kbps": 750,
        "avg_lat_ms": 170,
        "tower_count": 3,
        "nearest_tower_km": 3.2
    },
    {
        "area_name": "Yatiyanthota",
        "district": "Kegalle",
        "lat": 7.020,
        "lon": 80.300,
        "avg_d_kbps": 5200,
        "avg_u_kbps": 1000,
        "avg_lat_ms": 100,
        "tower_count": 4,
        "nearest_tower_km": 2.0
    }
])

network_df = pd.concat([network_df, new_rows], ignore_index=True)

network_df["network_suitability"] = network_df.apply(classify_network, axis=1)

network_df["sync_strategy"] = network_df["network_suitability"].map({
    "Good": "Full Sync",
    "Moderate": "Priority Sync Only",
    "Poor": "Offline Retry"
})

display(network_df)

,area_name,district,lat,lon,avg_d_kbps,avg_u_kbps,avg_lat_ms,tower_count,nearest_tower_km,network_suitability,sync_strategy
0,Neluwa,Galle,6.367,80.367,4200,900,140,4,2.1,Moderate,Priority Sync Only
1,Warukandeniya,Galle,6.350,80.390,1800,500,230,2,4.7,Moderate,Priority Sync Only
2,Tinniyawala,Kalutara,6.450,80.170,6500,1200,80,5,1.5,Good,Full Sync
3,Kalawana,Ratnapura,6.533,80.400,3500,750,170,3,3.2,Moderate,Priority Sync Only
4,Yatiyanthota,Kegalle,7.020,80.300,5200,1000,100,4,2.0,Good,Full Sync
5,Kalawana,Ratnapura,6.533,80.400,3500,750,170,3,3.2,Moderate,Priority Sync Only
6,Yatiyanthota,Kegalle,7.020,80.300,5200,1000,100,4,2.0,Good,Full Sync


In [9]:
X = network_df[features]
y = network_df["network_suitability"]

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X, y)

print("Model retrained successfully with updated data")

Model retrained successfully with updated data


In [10]:
predict_for_area("Kalawana")

,Area,Predicted Network Suitability,Confidence,Recommended Sync Strategy,Download Speed kbps,Upload Speed kbps,Latency ms,Tower Count,Nearest Tower km
0,Kalawana,Moderate,99.00%,Priority Sync Only - sync urgent collection re...,3500,750,170,3,3.2


In [11]:
predict_for_area("Yatiyanthota")

,Area,Predicted Network Suitability,Confidence,Recommended Sync Strategy,Download Speed kbps,Upload Speed kbps,Latency ms,Tower Count,Nearest Tower km
0,Yatiyanthota,Good,91.00%,Full Sync - sync all pending records,5200,1000,100,4,2.0


In [12]:
predict_for_area("Neluwa")

,Area,Predicted Network Suitability,Confidence,Recommended Sync Strategy,Download Speed kbps,Upload Speed kbps,Latency ms,Tower Count,Nearest Tower km
0,Neluwa,Moderate,90.00%,Priority Sync Only - sync urgent collection re...,4200,900,140,4,2.1


In [13]:
predict_for_area("Warukandeniya")

,Area,Predicted Network Suitability,Confidence,Recommended Sync Strategy,Download Speed kbps,Upload Speed kbps,Latency ms,Tower Count,Nearest Tower km
0,Warukandeniya,Moderate,100.00%,Priority Sync Only - sync urgent collection re...,1800,500,230,2,4.7


In [14]:
predict_for_area("Tinniyawala")

,Area,Predicted Network Suitability,Confidence,Recommended Sync Strategy,Download Speed kbps,Upload Speed kbps,Latency ms,Tower Count,Nearest Tower km
0,Tinniyawala,Good,90.00%,Full Sync - sync all pending records,6500,1200,80,5,1.5


In [15]:
predict_for_area("Kalawana")

,Area,Predicted Network Suitability,Confidence,Recommended Sync Strategy,Download Speed kbps,Upload Speed kbps,Latency ms,Tower Count,Nearest Tower km
0,Kalawana,Moderate,99.00%,Priority Sync Only - sync urgent collection re...,3500,750,170,3,3.2
